In [ ]:
# ------------------------------------------------------------
# This script randomly samples a subset of images and corresponding
# annotations from a full COCO-format dataset and saves the result
# as a smaller JSON file.
#
# Purpose:
# - Useful for quick experiments, debugging, or low-resource training.
#
# Inputs:
# - json_path: Path to the original COCO JSON file
# - output_path: Path to save the new sampled subset
# - sample_size: Number of images to randomly select
#
# Output:
# - A new JSON file containing the sampled images, annotations, 
#   and original category definitions.
# ------------------------------------------------------------

import json
import random
import os

def sample_coco_dataset(json_path, output_path, sample_size):
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # gain all image information
    images = coco_data['images']
    annotations = coco_data['annotations']

    # randomly choose
    sampled_images = random.sample(images, sample_size)
    sampled_image_ids = set(img['id'] for img in sampled_images)

    # filter annotations
    sampled_annotations = [ann for ann in annotations if ann['image_id'] in sampled_image_ids]

    # build new JSON
    sampled_coco_data = {
        'images': sampled_images,
        'annotations': sampled_annotations,
        'categories': coco_data['categories']
    }

    # output to specific document
    with open(output_path, 'w') as f:
        json.dump(sampled_coco_data, f)

    print(f"Subset saved to {output_path} with {len(sampled_images)} images and {len(sampled_annotations)} annotations.")


if __name__ == "__main__":
    # configuration
    json_path = "/Users/sakuramomoko/Desktop/Fish_counting/cfc_train.json"  # original COCO JSON path
    output_path = "train_small_subset.json"    # output folder name
    sample_size = 500                         # the number of choosing images

    sample_coco_dataset(json_path, output_path, sample_size)

In [ ]:
# ------------------------------------------------------------------------
# This script converts a COCO-format dataset to YOLO-format annotations.
# It also copies the corresponding images to a specified YOLO directory.
#
# Key Steps:
# 1. Read annotations from a COCO-format JSON file
# 2. Convert bounding boxes to YOLO format (x_center, y_center, width, height)
# 3. Write YOLO-format .txt label files (one per image)
# 4. Copy each image from the source directory to the YOLO image directory
#
# Usage:
# - Make sure to provide the correct paths for JSON, image source, and output folders.
# - Only one class is assumed (e.g., fish with class ID 0).
#
# Inputs:
# - COCO JSON annotation file
# - Source image directory
#
# Outputs:
# - YOLO-format .txt files in `labels/`
# - Copied images in `images/`
# ------------------------------------------------------------------------

import json
import os
import shutil
from tqdm import tqdm

def coco_to_yolo_bbox(bbox, img_width, img_height):
    x, y, w, h = bbox
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    w /= img_width
    h /= img_height
    return [x_center, y_center, w, h]

def convert_coco_to_yolo_and_copy_images(
        json_path, image_source_dir, image_target_dir, label_target_dir):

    os.makedirs(image_target_dir, exist_ok=True)
    os.makedirs(label_target_dir, exist_ok=True)

    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    images_info = {img['id']: img for img in coco_data['images']}
    annotations = coco_data['annotations']

    # create a dict： image_id -> annotations
    image_to_anns = {}
    for ann in annotations:
        image_id = ann['image_id']
        if image_id not in image_to_anns:
            image_to_anns[image_id] = []
        image_to_anns[image_id].append(ann)

    for image_id, image_info in tqdm(images_info.items(), desc='Processing images'):
        file_name = image_info['file_name']
        width = image_info['width']
        height = image_info['height']

        #anns = image_to_anns.get(image_info['id'], [])
        #if not anns:
        #    continue 

        src_img_path = os.path.join(image_source_dir, file_name)
        dst_img_path = os.path.join(image_target_dir, file_name)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
        else:
            print(f'Warning: Image not found - {src_img_path}')

        # write into corresponding YOLO txt file
        anns = image_to_anns.get(image_id, [])
        label_file_path = os.path.join(label_target_dir, file_name.replace('.jpg', '.txt'))
        with open(label_file_path, 'w') as lf:
            for ann in anns:
                yolo_bbox = coco_to_yolo_bbox(ann['bbox'], width, height)
                line = f"0 {' '.join(map(str, yolo_bbox))}\n"  # class only has fish，id=0
                lf.write(line)


if __name__ == "__main__":
    json_path = '/Users/sakuramomoko/Desktop/Fish_counting/train_small_subset.json'  # input JSON
    image_source_dir = '/Users/sakuramomoko/Desktop/Fish_counting/cfc_train'        # image original path
    image_target_dir = '/Users/sakuramomoko/Desktop/Fish_counting/yolo_images/train/images' # target image path
    label_target_dir = '/Users/sakuramomoko/Desktop/Fish_counting/yolo_images/train/labels' # labels image path

    convert_coco_to_yolo_and_copy_images(
        json_path, image_source_dir, image_target_dir, label_target_dir)

In [ ]:
# ------------------------------------------------------------------------
# This script creates a YOLO-format validation subset from a COCO-format dataset.
# It randomly samples a given number of images and:
#   1. Converts bounding boxes to YOLO format (x_center, y_center, width, height)
#   2. Writes YOLO .txt label files (one per image)
#   3. Copies the sampled images to a target directory
#
# Parameters:
# - json_path: COCO-format annotation file for the validation set
# - image_source_dir: path to the original images
# - image_target_dir: path to save copied images (YOLO val/images/)
# - label_target_dir: path to save label .txt files (YOLO val/labels/)
# - sample_size: number of validation images to extract (default = 500)
#
# Note:
# - Assumes a single class with class ID = 0 (e.g., fish)
# - If fewer than sample_size images exist, it samples all of them
# ------------------------------------------------------------------------

import json
import os
import shutil
import random
from tqdm import tqdm

def coco_to_yolo_bbox(bbox, img_width, img_height):
    x, y, w, h = bbox
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    w /= img_width
    h /= img_height
    return [x_center, y_center, w, h]

def convert_val_subset_and_copy(
        json_path, image_source_dir, image_target_dir, label_target_dir, sample_size=500):

    os.makedirs(image_target_dir, exist_ok=True)
    os.makedirs(label_target_dir, exist_ok=True)

    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    images_info = coco_data['images']
    annotations = coco_data['annotations']

    # Randomly extract some val samples
    sampled_images = random.sample(images_info, min(sample_size, len(images_info)))
    sampled_image_ids = set(img['id'] for img in sampled_images)

    # Filter annotations based on samples
    image_to_anns = {}
    for ann in annotations:
        if ann['image_id'] in sampled_image_ids:
            image_id = ann['image_id']
            if image_id not in image_to_anns:
                image_to_anns[image_id] = []
            image_to_anns[image_id].append(ann)

    for img_info in tqdm(sampled_images, desc='Processing val images'):
        file_name = img_info['file_name']
        width = img_info['width']
        height = img_info['height']

        #anns = image_to_anns.get(img_info['id'], [])
        #if not anns:
        #    continue 

        src_img_path = os.path.join(image_source_dir, file_name)
        dst_img_path = os.path.join(image_target_dir, file_name)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
        else:
            print(f'Warning: Image not found - {src_img_path}')

        anns = image_to_anns.get(img_info['id'], [])
        label_file_path = os.path.join(label_target_dir, file_name.replace('.jpg', '.txt'))
        with open(label_file_path, 'w') as lf:
            for ann in anns:
                yolo_bbox = coco_to_yolo_bbox(ann['bbox'], width, height)
                line = f"0 {' '.join(map(str, yolo_bbox))}\n"
                lf.write(line)

if __name__ == "__main__":
    json_path = '/Users/sakuramomoko/Desktop/Fish_counting/cfc_channel_test.json'  # val JSON file
    image_source_dir = '/Users/sakuramomoko/Desktop/Fish_counting/cfc_channel_test'  # Image source path
    image_target_dir = '/Users/sakuramomoko/Desktop/Fish_counting/yolo_images/val/images'   # Target val Image path
    label_target_dir = '/Users/sakuramomoko/Desktop/Fish_counting/yolo_images/val/labels'   # val label path

    convert_val_subset_and_copy(
        json_path, image_source_dir, image_target_dir, label_target_dir, sample_size=500)  #  sample_size